# Notebook 03 — Retriever Denso (TF-IDF + KNN Cosseno)

**Disciplina:** Inteligência Artificial — FACOM/UFMS — 2026/1
**Tema:** IA para Reconhecimento de Vocalizações de Psittacidae

Implementa um **retriever denso** baseado em TF-IDF vetorial com similaridade de cosseno.
Diferente do BM25 (term-matching), este modelo representa documentos e queries em
espaço vetorial de alta dimensão, permitindo capturar coocorrência de termos via n-gramas.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, "..")

from src.utils import load_corpus, load_queries, write_trec_run
from src.retrievers import TFIDFKNNRetriever


## 1. Carregamento do corpus

In [2]:
docs = load_corpus("../data/corpus.jsonl")
print(f"Documentos: {len(docs)}")


Documentos: 1092


## 2. Vetorização TF-IDF

Parâmetros escolhidos:
- **max_features = 50.000** — vocabulário balanceia cobertura e dimensão
- **ngram_range = (1, 2)** — unigrams e bigrams capturam frases como "bird call", "deep learning"
- **sublinear_tf = True** — aplica log(1 + tf), reduz efeito de termos muito frequentes


In [3]:
retriever = TFIDFKNNRetriever(docs, max_features=50_000, ngram_range=(1, 2))
vocab_size = len(retriever._vec.vocabulary_)
print(f"Vocabulário TF-IDF: {vocab_size} features")
print(f"Matriz doc-term: {retriever._matrix.shape}")


Vocabulário TF-IDF: 50000 features
Matriz doc-term: (1092, 50000)


## 3. Busca de exemplo

In [4]:
query = "deep learning for parrot call recognition mel spectrogram"
results = retriever.search(query, k=10)

print(f"Top-10 para: {query!r}\n")
for rank, (doc_id, score) in enumerate(results, 1):
    doc = next(d for d in docs if d["arxiv_id"] == doc_id)
    print(f"{rank:2}. [{score:.4f}] {doc['title'][:75]}")
    print(f"    {doc_id} | {doc.get('primary_category','')} | {doc.get('published','')}")


Top-10 para: 'deep learning for parrot call recognition mel spectrogram'

 1. [0.1697] Bird Call Recognition using Deep Convolutional Neural Network, ResNet-50
    W2944638455 | Animal Vocal Communication and Behavior | 2018
 2. [0.1193] Bird Call Recognition using Acoustic based Feature Selection approach in Ma
    W4391092908 | Animal Vocal Communication and Behavior | 2023
 3. [0.1024] An EfficientNet-based Ensemble for Bird-Call Recognition with Enhanced Nois
    W4386479244 | Speech and Audio Processing | 2023
 4. [0.0988] Classification of Sound using Convolutional Neural Networks
    W4360585215 | Music and Audio Processing | 2022
 5. [0.0959] Implementation of Constant-Q Transform (CQT) and Mel Spectrogram to convert
    W3201280717 | Animal Vocal Communication and Behavior | 2021
 6. [0.0922] Towards Citizen Science for Smart Cities: A Framework for a Collaborative G
    2103.16988 | cs.SD | 2021
 7. [0.0912] Improved Bird Sound Classification Based on Deep Cascade Feature
   

## 4. Geração do arquivo de run

In [5]:
queries = load_queries("../eval/queries.tsv")
runs_dir = Path("runs")
runs_dir.mkdir(exist_ok=True)

results_all = {}
for _, row in queries.iterrows():
    results_all[row["qid"]] = retriever.search(row["text"], k=100)

write_trec_run(results_all, runs_dir / "knn_tfidf.trec", system_name="knn_tfidf")
print("Run salva em runs/knn_tfidf.trec")


Run salva em runs/knn_tfidf.trec


## 5. Comparação BM25 vs TF-IDF KNN (top-5 para a mesma query)

In [6]:
from src.retrievers import BM25Retriever

bm25 = BM25Retriever(docs)

q = "deep learning for parrot call recognition mel spectrogram"
print(f"Query: {q!r}\n")

bm25_res = bm25.search(q, k=5)
tfidf_res = retriever.search(q, k=5)

print("--- BM25 top-5 ---")
for r, (doc_id, sc) in enumerate(bm25_res, 1):
    doc = next(d for d in docs if d["arxiv_id"] == doc_id)
    print(f"  {r}. [{sc:.3f}] {doc['title'][:70]}")

print("\n--- TF-IDF KNN top-5 ---")
for r, (doc_id, sc) in enumerate(tfidf_res, 1):
    doc = next(d for d in docs if d["arxiv_id"] == doc_id)
    print(f"  {r}. [{sc:.4f}] {doc['title'][:70]}")


Query: 'deep learning for parrot call recognition mel spectrogram'



--- BM25 top-5 ---
  1. [12.533] Classification of Sound using Convolutional Neural Networks
  2. [11.701] Vocal individuality, but not stability, in wild palm cockatoos (<i>Pro
  3. [11.400] A Novel Deep Learning Approach for Classification of Bird Sound Using 
  4. [11.053] Accounting for both automated recording unit detection space and signa
  5. [10.818] Contact calls of island Brown-throated Parakeets exhibit both characte

--- TF-IDF KNN top-5 ---
  1. [0.1697] Bird Call Recognition using Deep Convolutional Neural Network, ResNet-
  2. [0.1193] Bird Call Recognition using Acoustic based Feature Selection approach 
  3. [0.1024] An EfficientNet-based Ensemble for Bird-Call Recognition with Enhanced
  4. [0.0988] Classification of Sound using Convolutional Neural Networks
  5. [0.0959] Implementation of Constant-Q Transform (CQT) and Mel Spectrogram to co


## 6. Discussão

**TF-IDF vetorial vs BM25:**
- TF-IDF KNN considera a frequência global dos termos no vocabulário (IDF) e representa
  o documento como vetor ponderado; a similaridade de cosseno é invariante ao comprimento.
- BM25 é mais adequado para coleções com documentos de tamanho variável por conta da
  normalização explícita de comprimento; em coleções de abstracts (tamanho uniforme),
  as diferenças tendem a ser menores.
- O TF-IDF com bigrams captura colocações relevantes ("parrot call", "mel spectrogram")
  que o BM25 word-level não vê diretamente.

**Limitação comum:** ambos são modelos de "bag-of-words"; não capturam semântica
distributiva (e.g., "psittacine" ≈ "parrot"). O próximo passo seria usar embeddings
densos (sentence-transformers, SPECTER2), que vai além do escopo deste trabalho.
